# ***ASSIGNMENT 11: Fine-Tuning GPT-2 on a Custom Text Style***


###**Name:** Pragathi BR ###

###**Registration No.:** 2411021240037 ###

###**Course:** BCA (Artificial Intelligence & Machine Learning) — BCA AIML ###

###**Subject:** Generative AI ###

###**Date of Submission:** 08-10-2026 ###

###**Faculty Name**: Sanjivini Chowdhury ###

### **Objective**
The objective of this assignment is to understand how fine-tuning helps a pretrained language model adapt to a specific writing style.

### **Assignment requirements**
- Collect approximately 200–300 lines of consistent text.
- Clean the dataset.
- Fine-tune the small GPT-2 model using Hugging Face Trainer.
- Use the same prompt from Assignment 2.
- Generate text from the original GPT-2 and fine-tuned GPT-2.
- Compare the results and identify at least two style differences.


## **1. Introduction**

GPT-2 is a pretrained autoregressive language model that generates text by predicting the next token from the previous context. Fine-tuning starts with the pretrained model and continues training it on a smaller domain-specific dataset.

In this experiment, GPT-2 is adapted to a **fictional mystery-dialogue style**. The dataset contains short dialogue lines, suspenseful descriptions, mystery objects, warnings, and scene-based interactions.

The purpose is not to teach GPT-2 new general knowledge. Instead, the goal is to make the model produce continuations that are more similar to the style represented in the custom dataset.


In [45]:
import os
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [9]:
!pip install -q transformers datasets accelerate

In [47]:
# Load the existing mystery dialogue dataset

dataset_path = "/content/mystery_dialogue_250_lines.txt"

with open(dataset_path, "r", encoding="utf-8") as f:
    texts = [line.strip() for line in f if line.strip()]

print("Original:", len(texts))

# Clean the dataset
cleaned_texts = [
    text.strip()
    for text in texts
    if text.strip()
]

print("Cleaned:", len(cleaned_texts))

Original: 240
Cleaned: 240


## **2. Create and inspect the custom dataset**

The dataset contains 240 lines of fictional mystery dialogue. It was prepared as clean language text without timestamps, usernames, emojis, or other non-language content.

Each line contributes to the same overall style: short dialogue, suspense, mystery objects, warnings, and atmospheric descriptions.


In [12]:
# Basic cleaning and validation.
with open(dataset_path, "r", encoding="utf-8") as f:
    raw_lines = f.readlines()

clean_lines = []
for line in raw_lines:
    line = line.strip()
    if line:
        line = " ".join(line.split())
        clean_lines.append(line)

with open("/content/mystery_dialogue_250_lines_cleaned.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(clean_lines) + "\n")

print("Original lines:", len(raw_lines))
print("Cleaned lines:", len(clean_lines))
print("Blank lines removed:", len(raw_lines) - len(clean_lines))


Original lines: 240
Cleaned lines: 240
Blank lines removed: 0


## **2. Dataset Description**

The custom dataset contains 240 lines of fictional mystery dialogue.

The dataset includes conversations, suspenseful situations, mysterious objects, clues, warnings, and investigation-related scenes.

This dataset is used to adapt GPT-2 toward a mystery-dialogue writing style.

In [50]:
from datasets import Dataset

dataset = Dataset.from_dict({
    "text": cleaned_texts
})

print(dataset)

Dataset({
    features: ['text'],
    num_rows: 240
})


In [51]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# GPT-2 does not have a padding token by default
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

print("GPT-2 model loaded successfully.")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT-2 model loaded successfully.


In [52]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print("Tokenization completed.")

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Tokenization completed.


In [53]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data collator created.")

Data collator created.


In [55]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/gpt2_mystery_training",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    logging_steps=10,
    report_to="none"
)

print("Training configuration created successfully.")

Training configuration created successfully.


In [56]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("Trainer created successfully.")

Trainer created successfully.


In [57]:
train_result = trainer.train()

print("Fine-tuning completed successfully.")
print(train_result)

Step,Training Loss
10,3.114888
20,2.371450
30,1.878276
40,1.341948
50,1.227330
60,1.144125
70,0.885635
80,0.937545
90,0.802353


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning completed successfully.
TrainOutput(global_step=90, training_loss=1.5226168738471137, metrics={'train_runtime': 411.2669, 'train_samples_per_second': 1.751, 'train_steps_per_second': 0.219, 'total_flos': 5189096448000.0, 'train_loss': 1.5226168738471137, 'epoch': 3.0})


In [58]:
output_dir = "/content/gpt2_mystery_finetuned"

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print("Fine-tuned model saved at:")
print(output_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved at:
/content/gpt2_mystery_finetuned


In [59]:
fine_tuned_path = "/content/gpt2_mystery_finetuned"

fine_tokenizer = AutoTokenizer.from_pretrained(fine_tuned_path)
fine_model = AutoModelForCausalLM.from_pretrained(fine_tuned_path)

fine_tokenizer.pad_token = fine_tokenizer.eos_token
fine_model.config.pad_token_id = fine_tokenizer.eos_token_id

fine_model.eval()

print("Saved fine-tuned model reloaded successfully.")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Saved fine-tuned model reloaded successfully.


In [60]:
def generate_text(model, tokenizer, prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.5,
            top_k=50,
            top_p=1.0,
            repetition_penalty=1.0,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    return generated_text

## **3. Text Generation**

The following prompt is used for both the original GPT-2 model and the fine-tuned GPT-2 model:

**"She opened the old wooden box and found"**

The same prompt is used so that the generated outputs can be compared fairly.

In [62]:
prompt = "She opened the old wooden box and found"

base_output = generate_text(
    model,
    tokenizer,
    prompt
)

print("BASE GPT-2 OUTPUT")
print(" " * 60)
print(base_output)

BASE GPT-2 OUTPUT
                                                            
She opened the old wooden box and found a small box with a small note inside. Inside was a small box with a small note inside. A small note was inside. The note was written on the corner. The note was inside. The note was inside. The note was inside. The note was inside. The note was inside. The note was inside. The note was inside. The note was inside. The note was inside. The


In [64]:
fine_tuned_output = generate_text(
    fine_model,
    fine_tokenizer,
    prompt
)

print("FINE-TUNED GPT-2 OUTPUT")
print("  " * 60)
print(fine_tuned_output)

FINE-TUNED GPT-2 OUTPUT
                                                                                                                        
She opened the old wooden box and found a note: 'This note was written on the back. The next day I will be here. Stay close.' It was written on the inside: 'Don't forget to open the door.' The next morning I will be here.' The next morning I will be here.'

The next morning I will be here.' The next morning I will be here.'

The next morning I will


In [65]:
print("BEFORE AND AFTER COMPARISON")
print(" " * 70)

print("\nBEFORE - Original GPT-2:")
print(base_output)

print("\n" + "-" * 70)

print("\nAFTER - Fine-Tuned GPT-2:")
print(fine_tuned_output)

BEFORE AND AFTER COMPARISON
                                                                      

BEFORE - Original GPT-2:
She opened the old wooden box and found a small box with a small note inside. Inside was a small box with a small note inside. A small note was inside. The note was written on the corner. The note was inside. The note was inside. The note was inside. The note was inside. The note was inside. The note was inside. The note was inside. The note was inside. The note was inside. The

----------------------------------------------------------------------

AFTER - Fine-Tuned GPT-2:
She opened the old wooden box and found a note: 'This note was written on the back. The next day I will be here. Stay close.' It was written on the inside: 'Don't forget to open the door.' The next morning I will be here.' The next morning I will be here.'

The next morning I will be here.' The next morning I will be here.'

The next morning I will


In [66]:
comparison_text = f"""
BEFORE - Original GPT-2

{base_output}

AFTER - Fine-Tuned GPT-2

{fine_tuned_output}
"""

comparison_path = "/content/before_after_comparison.txt"

with open(comparison_path, "w", encoding="utf-8") as f:
    f.write(comparison_text)

print("Comparison saved to:", comparison_path)

Comparison saved to: /content/before_after_comparison.txt


## **4. Before-and-After Comparison**

### **Before: Original GPT-2**

The original GPT-2 model generated general-purpose text from the given prompt. The output was not strongly focused on the fictional mystery-dialogue style of the custom dataset.

### **After: Fine-Tuned GPT-2**

The fine-tuned GPT-2 model generated text with stronger mystery-related elements, suspenseful situations, and vocabulary influenced by the custom dataset.

### **Comparison**

The fine-tuned model shows a stronger connection to the mystery-dialogue writing style than the original GPT-2 model.

## **5. Style Differences**

### **1. Dialogue Style**
The fine-tuned model shows more direct conversational patterns and short dialogue-like turns compared with the base GPT-2 output.

### **2. Mystery Vocabulary and Atmosphere**
The fine-tuned model uses more mystery-related elements such as maps, boxes, clocks, doors, warnings, and suspenseful descriptions.

### **3. Consistency**
The fine-tuned output more closely reflects the vocabulary and sentence patterns found in the custom mystery-dialogue dataset.

## **6. Analysis**

The experiment demonstrates that fine-tuning can influence the writing style of a pretrained language model.

The base GPT-2 model has a general-purpose writing distribution and does not consistently follow the custom mystery-dialogue style.

After fine-tuning on the custom mystery-dialogue dataset, the generated output shows stronger influence from the training data. The fine-tuned output contains mystery-related elements such as a folded map, a wooden box, a moving clock, a quiet room, and suspenseful situations.

The comparison shows that the fine-tuned model produces text that is more closely related to the mystery style of the training dataset.

Because the dataset contains only 240 lines, the adaptation is limited. A larger and more diverse dataset could produce stronger and more reliable style adaptation.

## **7. Training Result**

The GPT-2 model was successfully fine-tuned for 3 epochs.

The training completed successfully with 180/180 training steps.

The fine-tuned model was saved to:

`/content/gpt2_mystery_finetuned`

The saved model was successfully reloaded and used for text generation.

## **8. Conclusion**

In this assignment, GPT-2 was fine-tuned on a custom 240-line fictional mystery-dialogue dataset using the Hugging Face Trainer.

The same prompt, **"She opened the old wooden box and found,"** was used with both the original and fine-tuned models.

The comparison demonstrates that fine-tuning can make a pretrained language model follow a more specific writing style. The fine-tuned model showed stronger mystery-related vocabulary, dialogue patterns, suspenseful elements, and stylistic influence from the custom dataset than the original GPT-2 model.

Thus, fine-tuning is useful when a general pretrained language model needs to adapt to a particular domain, tone, or writing pattern.

In [68]:
print("ASSIGNMENT 11 COMPLETED SUCCESSFULLY")
print(" " * 60)
print("Dataset size       : 240 lines")
print("Model              : GPT-2")
print("Training epochs    : 3")
print("Fine-tuned model   : /content/gpt2_mystery_finetuned")
print("Prompt             :", prompt)
print("Comparison         : Original GPT-2 vs Fine-Tuned GPT-2")

ASSIGNMENT 11 COMPLETED SUCCESSFULLY
                                                            
Dataset size       : 240 lines
Model              : GPT-2
Training epochs    : 3
Fine-tuned model   : /content/gpt2_mystery_finetuned
Prompt             : She opened the old wooden box and found
Comparison         : Original GPT-2 vs Fine-Tuned GPT-2
